In [0]:
%run ./set_auth_context.py

In [0]:
from pyspark.sql.functions import col, year, month, dayofmonth, lpad, to_timestamp

# Databricks Notebook
# Name   : nb_partition_raw_fact_tables
# Path   : /Repos/petiot/notebooks/shared/nb_partition_raw_fact_tables
# Purpose: Reads flat .json.gz fact files from raw/, writes them into
#          date-partitioned subfolders, then deletes the original flat file.
#          Run ONCE after uploading flat files via Azure Portal.
#
# How to upload via Azure Portal (no software needed):
#   1. Go to portal.azure.com
#   2. Navigate to your Storage Account → Containers → raw
#   3. Click Upload → select all 8 fact .json.gz files
#   4. Upload directly into raw/ (no subfolders needed at this point)
#
# Before running this notebook:
#   raw/
#     sales.json.gz
#     device_snapshots.json.gz
#     activity_snapshots.json.gz
#     health_snapshots.json.gz
#     feeding_events.json.gz
#     hydration_events.json.gz
#     device_failures.json.gz
#     firmware_deployments.json.gz
#     owners.json.gz            ← dimension, left untouched
#     pets.json.gz              ← dimension, left untouched
#     ...
#
# After running this notebook:
#   raw/
#     sales/year=2021/month=03/day=15/sales.json.gz
#     sales/year=2021/month=03/day=16/sales.json.gz
#     device_snapshots/year=2025/month=01/day=01/device_snapshots.json.gz
#     device_snapshots/year=2025/month=01/day=02/device_snapshots.json.gz
#     ...
#     owners.json.gz            ← untouched
#     pets.json.gz              ← untouched

# ── Widgets (receive from ADF or set interactively) ───────────────────────────
dbutils.widgets.text("storage_account", "petiot")
dbutils.widgets.text("raw_container",   "raw")

STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
RAW_CONTAINER   = dbutils.widgets.get("raw_container")

# 🔥 FIX: Use the direct Azure cloud URI instead of a DBFS mount point
RAW_BASE_PATH   = f"abfss://{RAW_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# ── Fact table → date/timestamp column used for partitioning ──────────────────
FACT_TABLES = {
    "sales":                 "sale_date",
    "device_snapshots":      "snapshot_timestamp",
    "activity_snapshots":    "snapshot_timestamp",
    "health_snapshots":      "snapshot_timestamp",
    "feeding_events":        "event_timestamp",
    "hydration_events":      "event_timestamp",
    "device_failures":       "failure_timestamp",
    "firmware_deployments":  "deployment_timestamp",
}

# ... [Keep imports and print statements] ...

print("=" * 65)
print("  nb_partition_raw_fact_tables")
print(f"  Target Cloud Path : {RAW_BASE_PATH}")
print(f"  Processing        : {len(FACT_TABLES)} fact tables")
print("=" * 65)

# ── Helper: rename Spark part files to clean table_name.json.gz ───────────────
def rename_and_clean(base_path, table):
    """
    Spark writes files as part-00000-xxxx.json.gz inside each partition folder.
    This function renames them to table_name.json.gz and removes Spark metadata.
    """
    try:
        year_dirs = [f.path for f in dbutils.fs.ls(base_path)
                     if f.name.startswith("year=")]
    except Exception:
        return

    for year_dir in year_dirs:
        try:
            month_dirs = [f.path for f in dbutils.fs.ls(year_dir)
                          if f.name.startswith("month=")]
        except Exception:
            continue
        for month_dir in month_dirs:
            try:
                day_dirs = [f.path for f in dbutils.fs.ls(month_dir)
                            if f.name.startswith("day=")]
            except Exception:
                continue
            for day_dir in day_dirs:
                try:
                    files = dbutils.fs.ls(day_dir)
                except Exception:
                    continue
                for f in files:
                    if f.name.startswith("part-") and ".json" in f.name:
                        # Rename to clean filename
                        clean_path = day_dir.rstrip("/") + f"/{table}.json"
                        dbutils.fs.mv(f.path, clean_path)
                    elif f.name.startswith("_"):
                        # Remove Spark job metadata files (_SUCCESS, _committed, etc.)
                        dbutils.fs.rm(f.path)

# ── Main loop ─────────────────────────────────────────────────────────────────
results = []

for table_name, ts_col in FACT_TABLES.items():

    # This will now correctly evaluate to: 
    # abfss://raw@petiot.dfs.core.windows.net/sales.json.gz
    flat_file  = f"{RAW_BASE_PATH}/{table_name}.json"
    out_path   = f"{RAW_BASE_PATH}/{table_name}"

    print(f"\n{'─'*65}")
    print(f"  [{list(FACT_TABLES.keys()).index(table_name)+1}/{len(FACT_TABLES)}]  {table_name}")
    print(f"  Partition column : {ts_col}")

    # ── 1. Verify flat file exists ────────────────────────────────────────────
    try:
        dbutils.fs.ls(flat_file)
    except Exception:
        print(f"  ⚠  Not found: {flat_file} — skipping.")
        results.append((table_name, "SKIPPED", 0, 0))
        continue

    # ── 2. Read flat .json.gz ─────────────────────────────────────────────────
    print(f"  Reading flat file ...")
    df = (spark.read
          .option("inferSchema", "true")
          .option("multiLine",   "true")
          .json(flat_file)
    )
    row_count = df.count()
    print(f"  Rows loaded : {row_count:,}")

    # ── 3. Derive year / month / day as zero-padded strings ──────────────────
    df = (df
          .withColumn("_ts",   to_timestamp(col(ts_col)))
          .withColumn("year",  year(col("_ts")).cast("string"))
          .withColumn("month", lpad(month(col("_ts")).cast("string"),    2, "0"))
          .withColumn("day",   lpad(dayofmonth(col("_ts")).cast("string"), 2, "0"))
          .drop("_ts")
    )

    # ── 4. Count distinct partitions (days) ───────────────────────────────────
    partition_count = df.select("year","month","day").distinct().count()
    print(f"  Distinct day partitions : {partition_count}")

    # ── 5. Write partitioned .json (one file per partition) ───────────────
    print(f"  Writing to {out_path}/year=*/month=*/day=*/")
    (df
     .repartition(col("year"), col("month"), col("day"))
     .write
     .partitionBy("year", "month", "day")
     .mode("overwrite")
     .json(out_path)
    )
    print(f"  Write complete.")

    # ── 6. Rename part files → table_name.json.gz, remove metadata ───────────
    print(f"  Renaming partition files to {table_name}.json ...")
    rename_and_clean(out_path, table_name)
    print(f"  Rename complete.")

    # ── 7. Delete original flat file ─────────────────────────────────────────
    print(f"  Deleting original flat file: {flat_file}")
    dbutils.fs.rm(flat_file, recurse=False)
    print(f"  ✓  {table_name} done.")

    results.append((table_name, "SUCCESS", row_count, partition_count))

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
print("  SUMMARY")
print(f"{'='*65}")
print(f"  {'Table':<30}  {'Status':<10}  {'Rows':>12}  {'Partitions':>12}")
print(f"  {'─'*30}  {'─'*10}  {'─'*12}  {'─'*12}")
for table, status, rows, parts in results:
    print(f"  {table:<30}  {status:<10}  {rows:>12,}  {parts:>12}")